In [1]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("data/global_ads_performance_dataset.csv")
df['date'] = pd.to_datetime(df['date'])

def blended_roas(sub):
    return sub['revenue'].sum() / sub['ad_spend'].sum()

results = {}

print("=== BLENDED ROAS THEO INDUSTRY x PLATFORM ===")
industry_check = df.pivot_table(index='industry', columns='platform', values=['ad_spend','revenue'], aggfunc='sum')
industry_roas = pd.DataFrame({
    p: industry_check['revenue'][p] / industry_check['ad_spend'][p]
    for p in df['platform'].unique()
}).round(2)
print(industry_roas)
industry_ranking_consistent = (industry_roas.rank(axis=1, ascending=False) == 
                                 industry_roas.mean().rank(ascending=False)).all(axis=1).sum()
print(f"\nSố ngành giữ đúng thứ hạng platform (TikTok>Meta>Google): {(industry_roas.idxmax(axis=1)=='TikTok Ads').sum()}/{len(industry_roas)} ngành có TikTok dẫn đầu")
results['by_industry'] = industry_roas.to_dict()

print("\n=== BLENDED ROAS THEO COUNTRY x PLATFORM ===")
country_check = df.pivot_table(index='country', columns='platform', values=['ad_spend','revenue'], aggfunc='sum')
country_roas = pd.DataFrame({
    p: country_check['revenue'][p] / country_check['ad_spend'][p]
    for p in df['platform'].unique()
}).round(2)
print(country_roas)
print(f"\nSố quốc gia có TikTok dẫn đầu: {(country_roas.idxmax(axis=1)=='TikTok Ads').sum()}/{len(country_roas)}")
results['by_country'] = country_roas.to_dict()

print("\n=== BLENDED ROAS: H1 vs H2 2024 ===")
df['half'] = np.where(df['date'] < '2024-07-01', 'H1_2024', 'H2_2024')
half_check = df.pivot_table(index='half', columns='platform', values=['ad_spend','revenue'], aggfunc='sum')
half_roas = pd.DataFrame({
    p: half_check['revenue'][p] / half_check['ad_spend'][p]
    for p in df['platform'].unique()
}).round(2)
print(half_roas)
results['by_half_year'] = half_roas.to_dict()

print("\n=== E-COMMERCE ONLY: BLENDED ROAS BY PLATFORM ===")
ecom = df[df['industry']=='E-commerce']
ecom_summary = ecom.groupby('platform').apply(
    lambda x: pd.Series({'blended_roas': blended_roas(x), 'total_spend': x['ad_spend'].sum(), 'total_revenue': x['revenue'].sum()})
).round(2)
print(ecom_summary)
results['ecommerce_only'] = ecom_summary.to_dict()

with open("dashboard_data/step5_robustness.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print("\n✅ Saved: dashboard_data/step5_robustness.json")

=== BLENDED ROAS THEO INDUSTRY x PLATFORM ===
            Google Ads  TikTok Ads  Meta Ads
industry                                    
E-commerce        3.70        8.36      4.97
EdTech            3.58        7.35      6.22
Fintech           3.70        6.38      4.83
Healthcare        3.12        7.68      6.51
SaaS              3.29        8.26      5.70

Số ngành giữ đúng thứ hạng platform (TikTok>Meta>Google): 5/5 ngành có TikTok dẫn đầu

=== BLENDED ROAS THEO COUNTRY x PLATFORM ===
           Google Ads  TikTok Ads  Meta Ads
country                                    
Australia        3.49        8.75      6.52
Canada           3.15        8.52      5.27
Germany          3.49        7.10      6.56
India            3.82        7.80      5.85
UAE              3.80        7.06      6.03
UK               3.70        7.61      5.12
USA              2.97        6.53      4.88

Số quốc gia có TikTok dẫn đầu: 7/7

=== BLENDED ROAS: H1 vs H2 2024 ===
         Google Ads  TikTok Ads  Meta

C:\Users\HP\AppData\Local\Temp\ipykernel_15888\1395176029.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ecom_summary = ecom.groupby('platform').apply(
